## Customer service automation (multi-agent):

## Agents:
### 1- Query Agent:Handles initial customer inquiries.
### 2- Resolution Agent:Solves technical or logistical issues.
### 3- Escalation Agent:Addresses complex queries oe escalates case to human representatives.

In [ ]:
from functools import partial
import operator
from typing import Annotated, Sequence, TypedDict, Literal
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_experimental.tools import PythonAstREPLTool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, BaseMessage
from pydantic import BaseModel
from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import create_react_agent


In [ ]:
# Define RouteResponse for customer service supervisor
class RouteResponseCS(BaseModel):
    next: Literal["Query_Agent", "Resolution_Agent", "Escalation_Agent", "FINISH"]

# Setup for customer service supervisor
members_cs = ["Query_Agent", "Resolution_Agent", "Escalation_Agent"]    
system_prompt_cs = f"You are a customer service supervisor managing agents:{','.join(members_cs)}."

# Create prompt template for the supervisor with correctly formatted options
prompt_cs = ChatPromptTemplate.from_template([
    ("system", system_prompt_cs),
    MessagesPlaceholder(variable_name="messages"),
    ("system", "Choose the next agent to act from {options}")
]).partial(options=str(members_cs))

# Define llm and supervisor function
llm = ChatOpenAI(model="gpt-4o-mini")

